# HEU MS2 Annotations

Across 2025_all10, 2026_modified, 2026_modified_lipidomics, 2026_max300, and 2026_targeted.

In [1]:
import os
import pickle
import json
import tqdm
import numpy as np
import subprocess
import pprint
import pandas as pd

from datetime import date
from asari.tools.ms2 import *
from asari.tools import entropy_search as ETS

Missing plottling libraries.


### .raw to .mzML

In [3]:
dir_path = (
    "/Users/chongj/Desktop/Li_Lab/Projects/HEU/2026-04-01_MS2/2026_modified_lipidomics/"
)

raw_2026_path = dir_path + "raw/"

modes = os.listdir(raw_2026_path)

print(modes)

['RPneg', 'RPpos']


In [ ]:
for m in modes:
    print(m)
    this_mode_path = dir_path + "raw/" + m
    this_out_path = dir_path + "mzML/" + m
    os.makedirs(this_out_path, exist_ok=True)
    print(this_out_path)

    raw_files = [f for f in os.listdir(this_mode_path) if f.endswith(".raw")]

    for f in raw_files:
        print(f)
        input_path = os.path.join(this_mode_path, f)
        result = subprocess.run(
            [
                "/Library/Frameworks/Mono.framework/Versions/Current/Commands/mono",
                "/Users/chongj/Downloads/ThermoRawFileParser1.4.4/ThermoRawFileParser.exe",
                f"-i={input_path}",
                f"-o={this_out_path}",
                "-f=1",
            ],
            capture_output=True,
            text=True,
        )
        print(f"Processing: {f}")
        if result.returncode == 0:
            print(f"  ✓ Done\n{result.stdout}")
        else:
            print(f"  ✗ Error\n{result.stderr}")

RPneg
/Users/chongj/Desktop/Li_Lab/Projects/HEU/2026-04-01_MS2/2026_modified_lipidomics//mzML/RPneg
HEU_RP_Neg_05.raw
HEU_RP_Neg_04.raw
HEU_RP_Neg_01.raw
HEU_RP_Neg_03.raw
HEU_RP_Neg_02.raw
Processing: HEU_RP_Neg_02.raw
  ✓ Done
2026-05-22 10:29:44 INFO Started parsing /Users/chongj/Desktop/Li_Lab/Projects/HEU/2026-04-01_MS2/2026_modified_lipidomics//raw/RPneg/HEU_RP_Neg_02.raw
2026-05-22 10:29:46 INFO Processing 2312 MS scans
10% 20% 30% 40% 50% 60% 70% 80% 90% 100% 

2026-05-22 10:29:53 INFO Finished parsing /Users/chongj/Desktop/Li_Lab/Projects/HEU/2026-04-01_MS2/2026_modified_lipidomics//raw/RPneg/HEU_RP_Neg_02.raw
2026-05-22 10:29:53 INFO Processing completed 0 errors, 0 warnings

RPpos
/Users/chongj/Desktop/Li_Lab/Projects/HEU/2026-04-01_MS2/2026_modified_lipidomics//mzML/RPpos
HEU_RP_Pos_03.raw
HEU_RP_Pos_02.raw
HEU_RP_Pos_01.raw
HEU_RP_Pos_05.raw
HEU_RP_Pos_04.raw
Processing: HEU_RP_Pos_04.raw
  ✓ Done
2026-05-22 10:30:28 INFO Started parsing /Users/chongj/Desktop/Li_Lab/Projec

In [6]:
# from mzML to jsons
this_workdir = dir_path + "mzML/"

for m in modes:
    this_mode_workdir = this_workdir + m
    print(this_mode_workdir)

    for f in [f for f in os.listdir(this_mode_workdir) if ".mzML" in f]:
        [ms1_spectra, ms2_spectra, others] = extract_all_spectra_form_file(
            os.path.join(this_mode_workdir, f)
        )
        with open(
            os.path.join(this_mode_workdir, f.replace(".mzML", "_MS1.json")), "w"
        ) as O:
            json.dump(ms1_spectra, O)
        with open(
            os.path.join(this_mode_workdir, f.replace(".mzML", "_MS2.json")), "w"
        ) as O:
            json.dump(ms2_spectra, O)

/Users/chongj/Desktop/Li_Lab/Projects/HEU/2026-04-01_MS2/2026_modified_lipidomics/mzML/RPneg
[Warning] Not index found and build_index_from_scratch is False
[Warning] Not index found and build_index_from_scratch is False
[Warning] Not index found and build_index_from_scratch is False
[Warning] Not index found and build_index_from_scratch is False
[Warning] Not index found and build_index_from_scratch is False
[Warning] Not index found and build_index_from_scratch is False
[Warning] Not index found and build_index_from_scratch is False
[Warning] Not index found and build_index_from_scratch is False
[Warning] Not index found and build_index_from_scratch is False
[Warning] Not index found and build_index_from_scratch is False
/Users/chongj/Desktop/Li_Lab/Projects/HEU/2026-04-01_MS2/2026_modified_lipidomics/mzML/RPpos
[Warning] Not index found and build_index_from_scratch is False
[Warning] Not index found and build_index_from_scratch is False
[Warning] Not index found and build_index_from

## MS2 Database Setup

In [2]:
ms2_pkl_neg = "/Users/chongj/Desktop/Li_Lab/annotation_sources/MSMS-Public_experimentspectra-neg-VS19.pkl"
entropy_search_neg = pickle.load(open(ms2_pkl_neg, "rb"))

pprint.pprint(entropy_search_neg[0])

{'CCS': '140.3921648',
 'COLLISIONENERGY': '50 V',
 'COMMENT': 'DB#=KO000046; origin=MassBank High Quality Mass Spectral Database',
 'FORMULA': 'C2H4O2',
 'INCHIKEY': 'QTBSBXVTEAMEQO-UHFFFAOYSA-N',
 'INSTRUMENTTYPE': 'LC-ESI-QQ',
 'IONMODE': 'Negative',
 'NAME': 'Acetate',
 'Num Peaks': '20',
 'ONTOLOGY': 'Carboxylic acids',
 'PRECURSORMZ': '59.01385294800001',
 'PRECURSORTYPE': '[M-H]-',
 'SMILES': 'CC(O)=O',
 'peaks': array([[3.2000000e+01, 6.5088756e-02],
       [3.8900002e+01, 1.1176857e-02],
       [3.9900002e+01, 3.6817882e-02],
       [4.1299999e+01, 9.0729780e-02],
       [4.1900002e+01, 5.6541748e-02],
       [4.3099998e+01, 8.5470090e-03],
       [4.3799999e+01, 1.9723866e-02],
       [5.0900002e+01, 1.7094018e-02],
       [5.2200001e+01, 3.9447732e-02],
       [5.3900002e+01, 4.5364890e-02],
       [5.5099998e+01, 7.1005918e-02],
       [5.6000000e+01, 1.2491782e-01],
       [5.6400002e+01, 1.6699539e-01],
       [5.7099998e+01, 2.4654832e-01]], dtype=float32),
 'precursor_m

In [3]:
ms2_pkl_pos = "/Users/chongj/Desktop/Li_Lab/annotation_sources/MSMS-Public_experimentspectra-pos-VS19.pkl"
entropy_search_pos = pickle.load(open(ms2_pkl_pos, "rb"))

pprint.pprint(entropy_search_pos[0])

{'COLLISIONENERGY': '50 V',
 'COMMENT': 'DB#=KO002858; origin=MassBank High Quality Mass Spectral Database',
 'FORMULA': 'C2H7N',
 'INCHIKEY': 'QUSNBJAOOMFDIB-UHFFFAOYSA-N',
 'INSTRUMENTTYPE': 'LC-ESI-QQ',
 'IONMODE': 'Positive',
 'NAME': 'Ethylamine',
 'Num Peaks': '22',
 'ONTOLOGY': 'Monoalkylamines',
 'PRECURSORMZ': '46.06512564399999',
 'PRECURSORTYPE': '[M+H]+',
 'SMILES': 'CCN',
 'peaks': array([[1.4100000e+01, 1.5218383e-01],
       [1.6000000e+01, 5.4329630e-02],
       [1.6700001e+01, 1.3042155e-01],
       [2.2000000e+01, 1.5218383e-01],
       [2.3100000e+01, 3.2567341e-02],
       [2.6100000e+01, 1.3042155e-01],
       [2.7500000e+01, 7.6091915e-02],
       [2.8600000e+01, 3.2567341e-02],
       [3.2299999e+01, 1.4137879e-01],
       [3.8599998e+01, 5.4329630e-02],
       [4.1299999e+01, 4.3524578e-02]], dtype=float32),
 'precursor_mz': 46.06512564399999}


In [4]:
mona_neg_pkl = "/Users/chongj/Desktop/Li_Lab/annotation_sources/MoNA-export-LC-MS-MS_Negative_Mode.pkl"
entropy_search_mona_neg = pickle.load(open(mona_neg_pkl, "rb"))

pprint.pprint(entropy_search_mona_neg[0])

{'db': 'MoNA',
 'id': 'KO000046',
 'inchikey': 'QTBSBXVTEAMEQO-UHFFFAOYSA-N',
 'mode': 'N',
 'name': 'Acetate',
 'peaks': array([[3.20000000e+01, 6.51558042e-02],
       [3.89000015e+01, 1.13314455e-02],
       [3.99000015e+01, 3.68271954e-02],
       [4.12999992e+01, 9.06515568e-02],
       [4.19000015e+01, 5.66572323e-02],
       [4.30999985e+01, 8.49858113e-03],
       [4.37999992e+01, 1.98300257e-02],
       [5.09000015e+01, 1.69971678e-02],
       [5.22000008e+01, 3.96600589e-02],
       [5.39000015e+01, 4.53257822e-02],
       [5.50999985e+01, 7.08215311e-02],
       [5.60000000e+01, 1.24645896e-01],
       [5.64000015e+01, 1.67138815e-01],
       [5.70999985e+01, 2.46458933e-01]], dtype=float32),
 'precursor_mz': 59.0}


In [5]:
mona_pos_pkl = "/Users/chongj/Desktop/Li_Lab/annotation_sources/MoNA-export-LC-MS-MS_Positive_Mode.pkl"
entropy_search_mona_pos = pickle.load(open(mona_pos_pkl, "rb"))

pprint.pprint(entropy_search_mona_pos[1])

{'db': 'MoNA',
 'id': 'KO002858',
 'inchikey': 'QUSNBJAOOMFDIB-UHFFFAOYSA-N',
 'mode': 'P',
 'name': 'Ethylamine',
 'peaks': array([[1.4100000e+01, 1.5217392e-01],
       [1.6000000e+01, 5.4347832e-02],
       [1.6700001e+01, 1.3043480e-01],
       [2.2000000e+01, 1.5217392e-01],
       [2.3100000e+01, 3.2608699e-02],
       [2.6100000e+01, 1.3043480e-01],
       [2.7500000e+01, 7.6086961e-02],
       [2.8600000e+01, 3.2608699e-02],
       [3.2299999e+01, 1.4130434e-01],
       [3.8599998e+01, 5.4347832e-02],
       [4.1299999e+01, 4.3478262e-02]], dtype=float32),
 'precursor_mz': 46.0}


## Run MS2 annotations

In [6]:
def search_ms2_files(mode_path, entropy_search, mona_search, param):
    results_dict = {}
    # for every cycle
    for f in [f for f in os.listdir(mode_path) if "_MS2.json" in f]:
        list_ms2_spectra = json.load(open(os.path.join(mode_path, f)))
        print(f"Loaded {len(list_ms2_spectra)} MS2 spectra from {f}")

        # MS Dial
        cleaned = ETS.clean_list_ms2spectra(list_ms2_spectra, entropy_search, param)
        results = search_ms2_spectra_entropy_cosine(cleaned, entropy_search, param)
        for r in results:
            r["db"] = "MS Dial"
        print(f"Found {len(results)} MS Dial matches in {f}")
        results_dict[f"{f}_MSDial"] = results

        # MoNA
        cleaned_mona = ETS.clean_list_ms2spectra(list_ms2_spectra, mona_search, param)
        results_mona = search_ms2_spectra_entropy_cosine(
            cleaned_mona, mona_search, param
        )
        for rm in results_mona:
            rm["db"] = "MoNA"
        print(f"Found {len(results_mona)} MoNA matches in {f}")
        results_dict[f"{f}_MoNA"] = results_mona

    return results_dict


def search_ms2_spectra_entropy_cosine(cleaned_spectra, entropy_search, params):
    results = []
    for precursor_mz, peaks, rtime, spid in tqdm.tqdm(
        cleaned_spectra, desc="Searching MS2 spectra"
    ):
        similarity, matched_num = entropy_search.identity_search(
            precursor_mz=precursor_mz,
            peaks=peaks,
            ms1_tolerance_in_da=params["mz_tol_ms1"],
            ms2_tolerance_in_da=params["mz_tol_ms2"],
            output_matched_peak_number=True,
        )
        # note this is to filter to similar enough for cosine similarity scoring since entropy is quick
        if np.max(similarity) > params["ms2_sim_tol"]:
            # get best entropy match
            best_idx = np.argmax(similarity)
            entropy_matched_entry = entropy_search[best_idx]
            entropy_score = similarity[best_idx]
            number_matched_peaks = matched_num[best_idx]
            # get cosine score
            cosine_matched_entry = entropy_search[best_idx]
            cosine_score = simple_cosine(
                peaks,
                entropy_matched_entry["peaks"],
                mz_tolerance=params["mz_tol_ms2"],
            )

            results.append(
                {
                    "spid": spid,
                    "precursor_mz": precursor_mz,
                    "rtime": rtime,
                    "peaks": peaks,
                    "entropy_matched_entry": entropy_matched_entry,
                    "entropy_score": entropy_score,
                    # 'cosine_matched_entry': cosine_matched_entry,
                    "cosine_score": cosine_score,
                    "number_matched_peaks": number_matched_peaks,
                }
            )
    return results


def simple_cosine(spec1, spec2, mz_tolerance=0.05):
    """
    Greedy cosine similarity: peaks are matched in descending order of
    intensity product, each peak used at most once.

    This is the most common cosine variant in MS/MS database search
    (equivalent to CosineGreedy in matchms).
    """
    s1 = np.array(spec1, dtype=float)
    s2 = np.array(spec2, dtype=float)

    norm1 = np.linalg.norm(s1[:, 1])
    norm2 = np.linalg.norm(s2[:, 1])
    if norm1 == 0 or norm2 == 0:
        return 0.0
    s1[:, 1] /= norm1
    s2[:, 1] /= norm2

    # Collect all candidate pairs within mz_tolerance, rank by intensity product
    candidates = []
    for i in range(len(s1)):
        for j in range(len(s2)):
            if abs(s1[i, 0] - s2[j, 0]) <= mz_tolerance:
                candidates.append((s1[i, 1] * s2[j, 1], i, j))
    candidates.sort(reverse=True)

    used1, used2 = set(), set()
    score = 0.0
    for prod, i, j in candidates:
        if i not in used1 and j not in used2:
            score += prod
            used1.add(i)
            used2.add(j)

    return min(score, 1.0)


def extract_db(comment):
    import re

    m = re.search(r"DB#=([^;]+)", comment or "")
    return m.group(1) if m else None

In [12]:
params = {
    "mz_tol_ms1": 0.01,
    "mz_tol_ms2": 0.01,
    "ms2_sim_tol": 0.2,  # intentionally low, to filter later
    "precursor_mz_offset": 1.6,
}

SCORE_CUTOFF = 0.5
MATCHED_PEAKS_CUTOFF = 2

years = [
    "2025",
    "2026_max300",
    "2026_modified_lipidomics",
    "2026_modified_regular",
    "2026_targeted_MS2",
]

modes = ["HILICneg", "HILICpos", "RPneg", "RPpos"]

ms2_dir = "/Users/chongj/Desktop/Li_Lab/Projects/HEU/2026-04-01_MS2/"

ms2_output_dir = ms2_dir + date.today().strftime("%Y-%m-%d") + "_ms2_annotations/"

In [9]:
this_year_search_results = {}

for y in tqdm.tqdm(years, desc="Years"):
    for m in modes:
        # set up ms2 libraries for search
        if "neg" in m:
            msdial_lib = entropy_search_neg
            mona_lib = entropy_search_mona_neg
        else:
            msdial_lib = entropy_search_pos
            mona_lib = entropy_search_mona_pos

        this_mode_path = ms2_dir + y + "/mzML/" + m + "/"

        if not os.path.isdir(this_mode_path):
            print(f"Skipping {y}/{m} - directory not found")
            continue

        this_year_search_results[f"{y}_{m}"] = search_ms2_files(
            this_mode_path, msdial_lib, mona_lib, params
        )


Years:   0%|          | 0/5 [00:00<?, ?it/s]

Loaded 2212 MS2 spectra from ID_02_MS2.json


Searching MS2 spectra: 100%|██████████| 2212/2212 [00:00<00:00, 8807.00it/s]


Found 371 MS Dial matches in ID_02_MS2.json


Searching MS2 spectra: 100%|██████████| 2212/2212 [00:00<00:00, 7817.77it/s]


...

Found 6 MoNA matches in RPneg_PRM_NCE15_MS2.json
Loaded 1335 MS2 spectra from RPpos_PRM_MS2.json


Searching MS2 spectra: 100%|██████████| 1335/1335 [00:00<00:00, 2207.86it/s]


Found 126 MS Dial matches in RPpos_PRM_MS2.json


Years: 100%|██████████| 5/5 [01:07<00:00, 13.41s/it]

Found 145 MoNA matches in RPpos_PRM_MS2.json


In [37]:
# ok here we have nested dicts (2 levels)
# first year + mode
keys = this_year_search_results.keys()
print(keys)

dict_keys(['2025_HILICneg', '2025_HILICpos', '2025_RPneg', '2025_RPpos', '2026_max300_HILICneg', '2026_max300_HILICpos', '2026_max300_RPneg', '2026_max300_RPpos', '2026_modified_lipidomics_RPneg', '2026_modified_lipidomics_RPpos', '2026_modified_regular_HILICneg', '2026_modified_regular_HILICpos', '2026_modified_regular_RPneg', '2026_modified_regular_RPpos', '2026_targeted_MS2_HILICneg', '2026_targeted_MS2_HILICpos', '2026_targeted_MS2_RPneg', '2026_targeted_MS2_RPpos'])


In [38]:
# second by file and database
this_year_search_results["2026_targeted_MS2_HILICneg"].keys()

dict_keys(['HILICneg_PRM_NCE25_MS2.json_MSDial', 'HILICneg_PRM_NCE25_MS2.json_MoNA', 'HILICneg_PRM_NCE15_MS2.json_MSDial', 'HILICneg_PRM_NCE15_MS2.json_MoNA', 'HILICneg_PRM_NCE40_MS2.json_MSDial', 'HILICneg_PRM_NCE40_MS2.json_MoNA'])

In [60]:
this_year_search_results["2026_targeted_MS2_HILICneg"][
    "HILICneg_PRM_NCE25_MS2.json_MSDial"
][0]

{'spid': 'sp209',
 'precursor_mz': 179.048004150391,
 'rtime': 14.9497716,
 'peaks': array([[5.05622787e+01, 7.30430195e-03],
        [5.90130882e+01, 5.58698416e-01],
        [6.35630798e+01, 7.78832193e-03],
        [6.38170929e+01, 7.28963455e-03],
        [6.66484833e+01, 7.71498540e-03],
        [9.29947128e+01, 9.77427065e-02],
        [9.69592209e+01, 7.56097911e-03],
        [9.71109390e+01, 8.61702301e-03],
        [1.00389923e+02, 6.87895110e-03],
        [1.02173325e+02, 6.71027740e-03],
        [1.07623680e+02, 7.70398509e-03],
        [1.21028564e+02, 2.25912668e-02],
        [1.35048264e+02, 6.54160380e-02],
        [1.38999130e+02, 2.06845216e-02],
        [1.45845230e+02, 9.70240124e-03],
        [1.50999786e+02, 9.86740831e-03],
        [1.51076233e+02, 8.22100602e-03],
        [1.52024658e+02, 1.58223212e-02],
        [1.58990509e+02, 9.34011936e-02],
        [1.60891785e+02, 1.94524713e-02],
        [1.66908020e+02, 1.08317807e-02]], dtype=float32),
 'entropy_matched

In [13]:
json_combined = []
pd_combined = []

# k is the year + mode
# v is the dict within k
for k, v in this_year_search_results.items():
    for file, matches in v.items():
        file_db = "MS Dial" if "_MSDial" in file else "MoNA"
        for match in matches:
            if match["number_matched_peaks"] >= MATCHED_PEAKS_CUTOFF and (
                match["cosine_score"] >= SCORE_CUTOFF
                or match["entropy_score"] >= SCORE_CUTOFF
            ):
                # infer db from file key if not stamped on the match (older search results)
                db = match.get("db") or file_db

                meta = match["entropy_matched_entry"]

                meta_lower = {mk.lower(): mv for mk, mv in meta.items()}

                db_id = (
                    extract_db(meta_lower.get("comment"))
                    if db == "MS Dial"
                    else meta_lower.get("id")
                )

                peaks = meta["peaks"]
                db_entry = {
                    **meta,
                    "peaks": peaks.tolist() if not isinstance(peaks, list) else peaks,
                }

                row = {
                    "year_mode": k,
                    "source_file": file,
                    "db": db,
                    "db_id": db_id,
                    "db_inchikey": meta_lower.get("inchikey"),
                    "db_smiles": meta_lower.get("smiles"),
                    "db_name": meta_lower.get("name"),
                    "db_precursor_mz": meta_lower.get(
                        "precursor_mz", meta_lower.get("precursormz")
                    ),
                    "exp_spid": match["spid"],
                    "exp_precursor_mz": match["precursor_mz"],
                    "exp_rtime": match["rtime"],
                    "exp_peaks": match["peaks"].tolist(),
                    "cosine_score": float(match["cosine_score"]),
                    "entropy_score": float(match["entropy_score"]),
                    "number_matched_peaks": int(match["number_matched_peaks"]),
                }

                pd_combined.append(row)
                json_combined.append({**row, "db_entry": db_entry})

pd_combined_df = pd.DataFrame(pd_combined)
pd_combined_df.head()

,year_mode,source_file,db,db_id,db_inchikey,db_smiles,db_name,db_precursor_mz,exp_spid,exp_precursor_mz,exp_rtime,exp_peaks,cosine_score,entropy_score,number_matched_peaks
0,2025_HILICneg,ID_02_MS2.json_MSDial,MS Dial,NUTRI-METAB-FEM-NEG001062,NGVLEQPKHLWZLN-UHFFFAOYSA-N,C1=CC(=C(C(=C1)OS(=O)(=O)O)O)O,pyrogallol_sulfate,204.981232,sp133,204.980347,24.129482,"[[41.00271987915039, 0.015400659292936325], [5...",0.747118,0.341986,2
1,2025_HILICneg,ID_02_MS2.json_MSDial,MS Dial,NUTRI-METAB-FEM-NEG001062,NGVLEQPKHLWZLN-UHFFFAOYSA-N,C1=CC(=C(C(=C1)OS(=O)(=O)O)O)O,pyrogallol_sulfate,204.981232,sp135,204.980347,24.370733,"[[79.95679473876953, 0.03478706628084183], [80...",0.870363,0.580462,3
2,2025_HILICneg,ID_02_MS2.json_MSDial,MS Dial,NUTRI-METAB-FEM-NEG000198,UIAFKZKHHVMJGS-UHFFFAOYSA-N,C1=CC(=C(C=C1O)O)C(=O)O,2_4_dihydroxybenzoic_acid,153.019332,sp144,153.018600,26.092644,"[[41.00279998779297, 0.28378036618232727], [41...",0.661045,0.606365,3
3,2025_HILICneg,ID_02_MS2.json_MSDial,MS Dial,NUTRI-METAB-FEM-NEG000237,AKEUNCKRJATALU-UHFFFAOYSA-N,C1=CC(=C(C(=C1)O)C(=O)O)O,2_6_dihydroxybenzoic_acid,153.019332,sp146,153.018600,26.436761,"[[41.0606803894043, 0.00986755546182394], [41....",0.971341,0.684022,2
4,2025_HILICneg,ID_02_MS2.json_MSDial,MS Dial,LQB00495,JAVWFBAAZSHHAD-KKILVFCFSA-N,CCCCCCCCCCCCCCCC(=O)OCC(COP([O-])(=O)OCC[N+](C...,PC 36:4,840.576006,sp153,840.571167,27.799070,"[[78.95862579345703, 0.049471985548734665], [8...",0.940276,0.881187,8


In [14]:
os.makedirs(ms2_output_dir, exist_ok=True)

csv_path = os.path.join(
    ms2_output_dir,
    f"{date.today().strftime('%Y-%m-%d')}_heu_combined_all_MS2_annotations.csv",
)
pd_combined_df.to_csv(csv_path, index=False)
print(f"Saved CSV  ({len(pd_combined_df)} rows)    → {csv_path}")

json_path = csv_path.replace(".csv", ".json")
with open(json_path, "w", encoding="utf-8") as O:
    json.dump(json_combined, O, ensure_ascii=False, indent=2)
print(f"Saved JSON ({len(json_combined)} entries) → {json_path}")

Saved CSV  (17461 rows)    → /Users/chongj/Desktop/Li_Lab/Projects/HEU/2026-04-01_MS2/2026-05-28_ms2_annotations/2026-05-28_heu_combined_all_MS2_annotations.csv
Saved JSON (17461 entries) → /Users/chongj/Desktop/Li_Lab/Projects/HEU/2026-04-01_MS2/2026-05-28_ms2_annotations/2026-05-28_heu_combined_all_MS2_annotations.json
